# NeoHuman — StyleGAN2-ADA 256×256 for Google Colab

**This notebook contains your 45 prepared images and its helper code.** Upload
this single `.ipynb` through **File → Upload notebook** in Google Colab.
Select **Runtime → Change runtime type → GPU** (T4 if offered). It uses Google's
hosted NVIDIA GPU, not your laptop's NVIDIA or Intel graphics. Your local 24-hour
GAN is independent and is not accessed or changed by this notebook.

Training and generation default to **OFF**, including when you choose Run all.
GPU availability and uninterrupted runtime are not guaranteed by Colab.

[NVIDIA paper](https://arxiv.org/abs/2006.06676) ·
[NVIDIA implementation](https://github.com/NVlabs/stylegan2-ada-pytorch) ·
[Colab runtime limits](https://research.google.com/colaboratory/faq.html)

This is an adapted, unexecuted cloud workflow. CPU checks were performed locally;
no Colab GPU training, Drive mount, or cloud checkpoint transfer has been tested.


## 1. Unpack the included code and 45 images

Run this once per new runtime. No model is loaded. The data is fitted and
white-padded to 256×256, preserving the full compositions. Images are unchanged
from the prepared local 256 dataset; no contrast edits have been applied.


In [ ]:
#@title Unpack the included dataset and helpers
from pathlib import Path
import base64, hashlib, io, json, os, shutil, signal, subprocess, sys, zipfile
assert Path('/content').is_dir(), 'Open this in a Google Colab hosted runtime.'
WORK = Path('/content/NeoHuman_ADA256_code')
WORK.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == '75de7d54d1e3d920c0c6c5d9a73d5b9908113b9a1b3802b8c73ea8e22f30101f', 'Embedded payload checksum failed.'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for member in z.infolist():
        target = (WORK / member.filename).resolve()
        assert target.is_relative_to(WORK.resolve()) and not member.is_dir()
        target.write_bytes(z.read(member))
del PAYLOAD, raw
print('Unpacked 45 images and helpers; no model started.')


## 2. Mount Google Drive and choose the separate project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/NeoHuman_ADA256_Colab')
PROJECT.mkdir(parents=True, exist_ok=True)
print('All durable snapshots will be saved under:', PROJECT)
print('Use this same Google account and project folder when reconnecting.')


## 3. Install the isolated runtime and fetch NVIDIA source/starting weights

This can take several minutes and downloads several GB of Python/CUDA packages.
It uses a separate Python environment; it does not replace Colab's notebook kernel.
The NVIDIA FFHQ256 starting weights are trained on faces, which is a domain
mismatch for this mixed drawing dataset. They are a starting point, not a promise
of image quality. Downloading does not run the model.


In [ ]:
# Paste this entire file into a new Colab cell, then run it.
# Repairs/retries only the isolated package installation; no model is loaded.
from pathlib import Path
import platform
import shutil
import subprocess
import sys

WORK = Path('/content/NeoHuman_ADA256_code')
assert Path('/content').is_dir(), 'Run this cell in your hosted Google Colab runtime.'
assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'The pinned packages support Python 3.10–3.13.'
WORK.mkdir(parents=True, exist_ok=True)
ENV = WORK / '.venv'
PYTHON = ENV / 'bin/python'
INSTALL_LOG = WORK / 'installation.log'


def install_command(argv):
    """Expose pip stderr in the notebook AND retain it in a local log."""
    tail = []
    with INSTALL_LOG.open('a', encoding='utf-8') as log:
        heading = '\nRUN: ' + ' '.join(map(str, argv)) + '\n'
        print(heading, flush=True)
        log.write(heading)
        with subprocess.Popen(list(map(str, argv)), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                tail.append(line.rstrip())
                tail = tail[-30:]
            result = process.wait()
    if result:
        raise RuntimeError(f'Installation command failed (exit {result}). '
                           f'Full output: {INSTALL_LOG}\nLast output:\n' + '\n'.join(tail))


print('Notebook Python:', sys.version)
print('Platform:', platform.platform(), '| machine:', platform.machine())
print(f'Free runtime disk: {shutil.disk_usage(WORK).free / 1024**3:.1f} GiB')
if not PYTHON.exists():
    install_command([sys.executable, '-m', 'venv', ENV])
install_command([PYTHON, '-c', 'import sys; print("Isolated Python:", sys.version)'])
install_command([PYTHON, '-m', 'ensurepip', '--upgrade'])
install_command([PYTHON, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir',
                 '--index-url', 'https://pypi.org/simple', 'pip'])
install_command([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '--progress-bar', 'off',
                 '--timeout', '120', '--retries', '3',
                 '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.7.1'])
install_command([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '--progress-bar', 'off',
                 '--index-url', 'https://pypi.org/simple',
                 'numpy==2.2.6', 'Pillow==11.2.1', 'scipy==1.15.3', 'click==8.2.1',
                 'requests==2.32.4', 'tqdm==4.67.1', 'psutil==7.0.0', 'ninja==1.11.1.4'])
install_command([PYTHON, '-m', 'pip', 'check'])
print('Package installation complete. Rerun notebook step 3 to fetch source/weights, then continue to step 4.')

VENDOR = WORK / 'vendor'
COMMIT = 'd72cc7d041b42ec8e806021a205ed9349f87c6a4'
if not VENDOR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NVlabs/stylegan2-ada-pytorch.git', str(VENDOR)], check=True)
    subprocess.run(['git', '-C', str(VENDOR), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(VENDOR), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
import urllib.request
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()
WEIGHTS_SHA = '7aa4ddeee38e007ce92a1a0bccd386fc6abba6b5c8692bdbe813d3f1987c0722'
WEIGHTS = WORK / 'ffhq256.pkl'
if not WEIGHTS.exists():
    url = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/transfer-learning-source-nets/ffhq-res256-mirror-paper256-noaug.pkl'
    temporary = WORK / 'ffhq256.partial'
    with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as f:
        shutil.copyfileobj(response, f)
    assert file_sha(temporary) == WEIGHTS_SHA, 'Starting-weight checksum mismatch.'
    temporary.replace(WEIGHTS)
assert file_sha(WEIGHTS) == WEIGHTS_SHA
print('Source and starting weights ready; no model loaded.')


## 4. Review settings and validate the plan on CPU

Default: **100 kimg** (100,000 image presentations, 12,500 batch updates), with a
**2-hour wall-time stop request per session**. These are separate limits; whichever
is reached first ends the segment. Time limits and manual stops are checked at the
next maintenance tick, so they can overshoot by a tick plus saving time.

Microbatch 4 processes four images together; effective batch 8 uses accumulation.
If a later GPU run runs out of memory, stop it, set microbatch to 2 or 1, then start
a new segment from a verified snapshot. There is no automatic retry. TF32 (where
supported), mixed precision and channels-last are enabled. The allocator budget
is at most 85% of VRAM, with at least 1 GiB launch-time headroom. Other apps/driver
allocations are outside that cap. Neither maximum utilization nor fitting is guaranteed.


In [ ]:
TOTAL_KIMG = 100
MAX_SESSION_HOURS = 2.0
MICROBATCH = 4
CFG = json.loads((WORK / 'config.json').read_text())
CFG.update(total_kimg=TOTAL_KIMG, session_hours=MAX_SESSION_HOURS, microbatch=MICROBATCH)
(WORK / 'config.json').write_text(json.dumps(CFG, indent=2))

def run_command(command, *args):
    argv = [str(PYTHON), '-u', str(WORK / 'cloud_runner.py'), command, '--project', str(PROJECT), *args]
    with subprocess.Popen(argv, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
        except KeyboardInterrupt:
            if command == 'train':
                (PROJECT / 'STOP').write_text('Manual notebook interrupt requested a graceful stop.\n')
                process.send_signal(signal.SIGINT)
                print('Stop requested. Keep this cell/runtime open while it saves to Drive.', flush=True)
                for line in process.stdout:
                    print(line, end='', flush=True)
            else:
                process.terminate()
                process.wait()
                raise
        code = process.wait()
    if code:
        raise RuntimeError(f'Command exited with code {code}; read the error above. No automatic retry.')

run_command('plan')
from IPython.display import Image, display
display(Image(filename=str(WORK / 'contact-sheet.jpg')))
print(CFG)


## 5. Inspect saved snapshots and recover a stale lock only if necessary

On a new runtime, rerun steps 1–4. The helper code/dataset live in temporary
`/content`; completed snapshots live in Drive. Do not use two notebooks to write
to this project at the same time. A disconnect can leave `writer.lock` behind.
Before releasing it, stop/delete the **old runtime**; never release a live writer.
This step is off by default and does not restart training.


In [ ]:
snapshots = sorted((PROJECT / 'runs').glob('*/*.verified.json'), key=lambda p: json.loads(p.read_text())['created'])
for receipt in snapshots:
    print(str(receipt).replace('.verified.json', '.pkl'))
if not snapshots:
    print('No verified training snapshots yet.')

CONFIRM_OLD_RUNTIME_STOPPED = False
CLEAR_PREVIOUS_STOP_REQUEST = False
owner_path = PROJECT / 'writer.lock/owner.json'
if owner_path.exists():
    owner = json.loads(owner_path.read_text())
    print('Existing writer lock:', owner)
    if CONFIRM_OLD_RUNTIME_STOPPED:
        run_command('release-lock', '--token', owner['token'], '--confirmation', 'I stopped the old Colab runtime')
if CLEAR_PREVIOUS_STOP_REQUEST:
    assert not (PROJECT / 'writer.lock').exists(), 'Stop the previous writer before clearing STOP.'
    (PROJECT / 'STOP').unlink(missing_ok=True)


## 6. Start training only when ready

Change `START_TRAINING` to True. For continuation, paste a verified Drive `.pkl`
path from step 5 into `RESUME_SNAPSHOT`; otherwise the first run uses FFHQ256.

**Continuation restores network weights only.** Optimizers, random state, ADA
adaptation and kimg counters restart. The kimg/time budgets apply to each new
segment, not the accumulated lifetime of the model. R3GAN `.pt` files are incompatible.

To stop: use **Interrupt execution once** (the cell's stop button) and wait for
`Verified on Drive` and `Exiting...`. Interrupting requests a save at the next
maintenance tick; it is not instantaneous. Do not disconnect/delete the runtime
to request a save. A second interrupt or forced termination may lose unsaved work.
If Colab kills the runtime unexpectedly, use the last verified Drive snapshot.

Snapshots are published after roughly each 1,000 image presentations and graceful
exit. Each segment retains its latest four and 10-kimg milestones. At 100 kimg,
allow roughly 5 GB of free Drive quota per segment; older segments remain saved.
Drive quota cannot be reliably inferred from the mount's free-disk report.


In [ ]:
START_TRAINING = False
RESUME_SNAPSHOT = None  # e.g. '/content/drive/MyDrive/NeoHuman_ADA256_Colab/runs/.../network-snapshot-....pkl'
if START_TRAINING:
    args = ['--execute']
    if RESUME_SNAPSHOT:
        args += ['--resume', RESUME_SNAPSHOT]
    run_command('train', *args)
else:
    print('Training is OFF. Change START_TRAINING only when ready.')


## 7. Inspect saved previews

In [ ]:
for folder in sorted((PROJECT / 'runs').glob('*')):
    print(folder)
    previews = sorted(folder.glob('fakes*.png'))
    if previews:
        display(Image(filename=str(previews[-1]), width=512))
    if (folder / 'result.json').exists():
        print((folder / 'result.json').read_text())


## 8. Generate native 256×256 images from a verified snapshot

Stop training first. Choose a `.pkl` from step 5, set `GENERATE = True`, and run
this cell. Use `DEVICE = 'cpu'` for slow generation on a CPU runtime when GPU time
is unavailable. Training itself does not fall back to CPU. Keep the same seeds
when comparing checkpoints. Images and their generation metadata are saved to Drive.


In [ ]:
GENERATE = False
SNAPSHOT = None
DEVICE = 'cuda'  # Change to 'cpu' for CPU generation.
SEEDS = '0,1,2,3'
if GENERATE:
    assert SNAPSHOT, 'Paste a verified snapshot path from step 5.'
    run_command('generate', '--execute', '--snapshot', SNAPSHOT, '--device', DEVICE, '--seeds', SEEDS)
else:
    print('Generation is OFF.')
